# CODE-M · Day 2 · Hands-on B
## Your ModelSEED drafts against the curated BiGG models

> **Student version.** Fill in the ✍️ cells. A complete version with all solutions is available from the instructors.

In KBase you built draft models of three bacteria with ModelSEED. Each of them already has a **curated** model in BiGG, built and checked by hand over years. This notebook does Steps 4 and 5 of the hands-on slides in Python: it loads the six models, translates between their identifiers, tests where they grow, compares their genes, and fills in the scorecard you'll bring to the 15:30 discussion.

| Organism | Your ModelSEED draft | Curated BiGG model |
|---|---|---|
| *Escherichia coli* K-12 MG1655 | built in KBase | iML1515 |
| *Bacillus subtilis* 168 | built in KBase | iYO844 |
| *Lactococcus lactis* MG1363 | built in KBase | iNF517 |

| Section | Slides | Time |
|---|---|---|
| 0 · Setup | | 3 min |
| 1 · Load the six models | Step 5 | 5 min |
| 2 · Same biology, different names | Step 5 | 7 min |
| 3 · How big is each model? | Scorecard | 5 min |
| 4 · What did gap-filling add? | Step 3 | 7 min |
| 5 · Who grows where? | Step 4 | 10 min |
| 6 · Genes shared with BiGG | Step 5 | 8 min |
| 7 · Your scorecard | Scorecard | 3 min |

**How to use this notebook**
- Run the cells in order with **Shift + Enter**.
- ✍️ marks a task for you: replace each `...` with your code.
- ✅ marks a checkpoint: if it prints a green tick, carry on. If it raises an error, the message tells you what to look at.
- 💬 marks a question to discuss with your neighbour.
- **Your own models.** The notebook starts with the instructors' KBase models in `../04_Models`. To use yours, export each model from KBase (click it in the Data panel, then the download icon; JSON or SBML both work), copy the files into `../04_Models` and change the file names in §1. Your numbers will differ a little from the ones quoted in the answers.

## 0 · Setup  ·  3 min
Run this cell to load the tools. The file `kbase_models.py`, in the same folder as this notebook, reads KBase model files and defines the media; you don't need to open it.

In [ ]:
import logging
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("cobra").setLevel(logging.ERROR)      # keep the output tidy

import pandas as pd
import cobra
from kbase_models import load_model, CARBON_D_GLUCOSE, AUXOMEDIA, M9_GLUCOSE_BIGG

cobra.Configuration().processes = 1
pd.set_option("display.max_colwidth", 70)
print("cobrapy", cobra.__version__, "| LP solver:", cobra.Configuration().solver.__name__.split("_")[0].split(".")[-1])

## 1 · Load the six models  ·  5 min

KBase saves models in its own format, not as SBML. `load_model()` converts them into cobrapy models. It also adds an exchange reaction for every compound outside the cell, as KBase does whenever it runs FBA. The BiGG models are SBML files from bigg.ucsd.edu; if one is missing from `../04_Models`, the cell downloads it. Loading takes a few seconds per model.

In [ ]:
MODELS = Path("../04_Models")
FILES = {
    "E. coli":     {"modelseed": "Escherichia_coli_str._K-12_substr._MG1655.RAST.mdl.json.zip",  "bigg": "iML1515.xml"},
    "B. subtilis": {"modelseed": "Bacillus_subtilis_subsp._subtilis_str._168.RAST.mdl.json.zip", "bigg": "iYO844.xml"},
    "L. lactis":   {"modelseed": "Lactococcus_lactis_subsp._cremoris_MG1363.RAST.mdl.json.zip",  "bigg": "iNF517.xml"},
}


def bigg_file(name):
    path = MODELS / name
    if not path.exists():                                  # not there yet: download it from BiGG
        import urllib.request
        urllib.request.urlretrieve(f"https://bigg.ucsd.edu/static/models/{name}", path)
    return path


ms, bigg = {}, {}
for org, f in FILES.items():
    ms[org] = load_model(MODELS / f["modelseed"])
    bigg[org] = cobra.io.read_sbml_model(str(bigg_file(f["bigg"])))
    print(f"{org:12s}  ModelSEED: {len(ms[org].reactions):5d} reactions    BiGG {bigg[org].id}: {len(bigg[org].reactions):5d} reactions")

In [ ]:
# ✅ Checkpoint
assert len(ms) == 3 and len(bigg) == 3
assert all(len(m.reactions) > 500 for m in list(ms.values()) + list(bigg.values()))
print("✅ Six models loaded")

## 2 · Same biology, different names  ·  7 min

ModelSEED and BiGG describe the same chemistry with different identifiers, so before comparing anything you need to know how each model names things. Start with glucose.

✍️ Complete `find_by_name()`: it should keep the metabolites whose name contains `text`, ignoring upper and lower case. (A metabolite's name is `m.name`.)

In [ ]:
def find_by_name(model, text):
    """Metabolites whose name contains `text` (ignoring case)."""
    hits = [m for m in model.metabolites if ...]        # ✍️ keep m if text appears in m.name (use .lower() on both)
    return pd.DataFrame([(m.id, m.name, m.compartment) for m in hits], columns=["id", "name", "compartment"])


ec_ms, ec_bigg = ms["E. coli"], bigg["E. coli"]
find_by_name(ec_ms, "D-glucose").head(8)

In [ ]:
find_by_name(ec_bigg, "D-glucose").head(8)

In [ ]:
# ✅ Checkpoint
hits = find_by_name(ec_ms, "glucose")
assert len(hits) and all("glucose" in name.lower() for name in hits.name), "find_by_name() should return only metabolites with 'glucose' in their name"
assert "cpd00027_e0" in set(hits.id), "find_by_name() should find cpd00027_e0 (D-Glucose outside the cell)"
assert "glc__D_e" in set(find_by_name(ec_bigg, "glucose").id), "find_by_name() should find glc__D_e in iML1515"
print("✅ Glucose is cpd00027 in ModelSEED and glc__D in BiGG")

The same translation applies to everything else. This table is the one on slide 8, read from the *E. coli* models.

In [ ]:
def biomass_id(model):
    return [r.id for r in model.reactions if r.objective_coefficient != 0][0]


pd.DataFrame({
    "ModelSEED (KBase)": ["cpd00027_e0", "EX_cpd00027_e0", biomass_id(ec_ms), ", ".join(sorted(g.id for g in ec_ms.genes)[:2]), ", ".join(ec_ms.compartments)],
    "BiGG (iML1515)":    ["glc__D_e", "EX_glc__D_e", biomass_id(ec_bigg), ", ".join(sorted(g.id for g in ec_bigg.genes)[:2]), ", ".join(ec_bigg.compartments)],
}, index=["glucose outside the cell", "glucose uptake", "biomass reaction", "genes", "compartments"])

Note the compartments: ModelSEED models have a cytosol (`c0`) and the outside (`e0`) only, while iML1515 also has the periplasm (`p`) between the two membranes of *E. coli*.

**Read the reaction, not its name.** Each ModelSEED biomass reaction is called *GramNegativeBiomass*, even in the two Gram-positive bacteria. That is a label inherited from KBase's templates. What matters is what the reaction contains:

In [ ]:
for org, model in ms.items():
    bio = model.reactions.get_by_id(biomass_id(model))
    wall = sorted({m.name for m in bio.metabolites if any(k in m.name.lower() for k in ("teichoic", "kdo2"))})
    print(f"{org:12s} '{bio.name}': {len(bio.metabolites)} components, e.g. {wall[:2]}")

The *E. coli* biomass contains lipid A precursors (KDO2-lipid IV<sub>A</sub>) of the Gram-negative outer membrane; the other two contain teichoic acids, which only Gram-positive cell walls have. KBase did choose the right template for each organism.

💬 Tomorrow you'll put two models in the same medium with COMETS. What has to be true of their identifiers for one to feed the other?

## 3 · How big is each model?  ·  5 min

These are the first three rows of the scorecard. One trap: KBase's output table counts reactions **without** the exchange reactions, while BiGG's totals include them. Here we count everything in the cobrapy model, as BiGG does, and also show the count without exchanges and other boundary reactions.

✍️ Complete `size()`. A reaction is a boundary reaction (exchange, demand or sink) if `r.boundary` is `True`.

In [ ]:
def size(model):
    return {"all reactions": len(model.reactions),
            "without exchanges": ...,          # ✍️ count the reactions that are NOT boundary reactions
            "metabolites": len(model.metabolites),
            "genes": ...}                      # ✍️ number of genes


sizes = pd.DataFrame({f"{org} · {name}": size(model)
                      for org in FILES for name, model in [("ModelSEED", ms[org]), (bigg[org].id, bigg[org])]}).T
sizes

In [ ]:
# ✅ Checkpoint
assert sizes.loc["E. coli · iML1515", "all reactions"] == 2712 and sizes.loc["E. coli · iML1515", "genes"] == 1516, "check size()"
assert (sizes["without exchanges"] < sizes["all reactions"]).all(), "'without exchanges' should be smaller than 'all reactions'"
print("✅ Sizes ready")

💬 Which drafts are bigger than their curated model, and which smaller? Why could an automatic draft have *more* reactions than a model curated by experts?

## 4 · What did gap-filling add?  ·  7 min

KBase records which reactions gap-filling added to make each draft grow on its medium; `load_model()` keeps that record in `reaction.notes["gapfilled"]`, together with the medium and the direction. (If you exported SBML instead of JSON, this record may be missing and the tables below stay empty.)

✍️ Complete `gapfilled()` so it keeps only the reactions that have a `"gapfilled"` entry in their notes.

In [ ]:
def gapfilled(model):
    rows = [(r.id, r.name, r.notes["gapfilled"], r.gene_reaction_rule or "(no gene)")
            for r in model.reactions if ...]                     # ✍️ keep r if "gapfilled" is in r.notes
    return pd.DataFrame(rows, columns=["reaction", "name", "added for", "genes"])


pd.DataFrame({org: {"gap-filled reactions": len(gapfilled(m)),
                    "reactions without a gene": sum(1 for r in m.reactions if not r.boundary and not r.genes)}
              for org, m in ms.items()})

In [ ]:
# ✅ Checkpoint
lactis_gf = gapfilled(ms["L. lactis"])
assert all("gapfilled" in ms["L. lactis"].reactions.get_by_id(rid).notes for rid in lactis_gf.reaction), "check the condition in gapfilled()"
print(f"✅ L. lactis: {len(lactis_gf)} reactions added by gap-filling")
lactis_gf.head(15)

💬 Look at the *L. lactis* list. Pick three reactions: real biology the annotation missed, or a patch to make the template's biomass possible? Why does *L. lactis* need the most gap-filling, even on the rich AuxoMedia?

## 5 · Who grows where?  ·  10 min

Slide 7 asked you to predict which models grow on which medium. Now test it. The two KBase media are defined in `kbase_models.py` with ModelSEED identifiers:
- **Carbon-D-Glucose**: a minimal medium, glucose plus salts;
- **AuxoMedia**: glucose plus amino acids, vitamins and other growth factors. KBase doesn't export the recipe, so this is an approximation.

For the BiGG models we use the same minimal glucose medium written in BiGG identifiers (M9 salts plus glucose).

In [ ]:
pd.DataFrame({"max uptake": CARBON_D_GLUCOSE,
              "name": {c: ec_ms.metabolites.get_by_id(c + "_e0").name for c in CARBON_D_GLUCOSE if c + "_e0" in ec_ms.metabolites}})

✍️ Complete `growth()`. `model.medium` takes a dictionary `{exchange reaction id: maximum uptake}`. Exchange reactions are called `EX_<compound><suffix>`: the suffix is `_e0` in ModelSEED models and `_e` in BiGG models. Skip compounds that have no exchange reaction in the model.

In [ ]:
def growth(model, medium, suffix):
    """Growth rate of `model` on `medium` ({compound id: maximum uptake})."""
    with model:                                    # every change is undone at the end of the block
        for ex in model.exchanges:                 # iNF517 forces some uptakes; here we allow them without forcing them
            if ex.upper_bound < 0:
                ex.upper_bound = 1000
        model.medium = ...                         # ✍️ {f"EX_{c}{suffix}": v for each compound c with uptake v, if that exchange is in the model}
        return model.slim_optimize(error_value=0)


growth_table = pd.DataFrame(
    [(org, growth(ms[org], CARBON_D_GLUCOSE, "_e0"), growth(ms[org], AUXOMEDIA, "_e0"), growth(bigg[org], M9_GLUCOSE_BIGG, "_e"))
     for org in FILES],
    columns=["organism", "ModelSEED · Carbon-D-Glucose", "ModelSEED · AuxoMedia", "BiGG · minimal glucose"]).set_index("organism").round(3)
growth_table

In [ ]:
# ✅ Checkpoint
assert growth_table.loc["E. coli", "ModelSEED · Carbon-D-Glucose"] > 0, "the E. coli draft was gap-filled on Carbon-D-Glucose: it should grow there"
assert growth_table.loc["L. lactis", "ModelSEED · AuxoMedia"] > 0, "the L. lactis draft was gap-filled on AuxoMedia: it should grow there"
print("✅ Growth tests done")

Compare **yes or no**, not the numbers: the media allow different amounts of glucose and the biomass reactions differ, so the growth rates aren't comparable between columns.

💬 Did your predictions from slide 7 hold? Why doesn't *L. lactis* grow on minimal glucose, in either model?

### 5.1 · Does the *B. subtilis* model know about trpC2?

The laboratory strain 168 carries the **trpC2** mutation, a small deletion in a tryptophan biosynthesis gene: it cannot grow without tryptophan.

✍️ Remove tryptophan (`cpd00065`) from AuxoMedia and test the draft. Minimal glucose contains no tryptophan at all, so iYO844 on minimal glucose is the same test for the curated model.

In [ ]:
no_trp = {c: v for c, v in AUXOMEDIA.items() if ...}        # ✍️ every compound except tryptophan, cpd00065
trp_test = {"ModelSEED draft, AuxoMedia without tryptophan": growth(ms["B. subtilis"], no_trp, "_e0"),
            "iYO844, minimal glucose (no tryptophan)": growth(bigg["B. subtilis"], M9_GLUCOSE_BIGG, "_e")}
pd.Series(trp_test).round(3)

In [ ]:
# ✅ Checkpoint
assert "cpd00065" not in no_trp and len(no_trp) == len(AUXOMEDIA) - 1, "no_trp should be AuxoMedia minus tryptophan only"
print("✅ Tryptophan test done")

## 6 · Genes shared with BiGG  ·  8 min

Genes are the most comparable part of two models: both use the locus tags of the genome annotation. Even so, they can be written differently (slide 8). Look at a few gene ids from each model:

In [ ]:
pd.DataFrame({org: {"ModelSEED genes": ", ".join(sorted(g.id for g in ms[org].genes)[:3]),
                    "BiGG genes": ", ".join(sorted(g.id for g in bigg[org].genes)[:3])} for org in FILES}).T

*E. coli* b-numbers and the *L. lactis* RefSeq tags (`LLMG_RS…`) are written the same way in both models, but *B. subtilis* tags are `BSU_00010` in the RefSeq genome and `BSU00010` in iYO844.

✍️ Complete `same_gene_id()` so that `BSU_00010` and `BSU00010`, or `b0001` and `B0001`, come out identical: remove underscores and use lower case. The table compares the genes shared with exact ids and with your normalised ids.

In [ ]:
def same_gene_id(gene_id):
    """Write a gene id in a common format: no underscores, lower case."""
    return ...                                       # ✍️


def shared_genes(draft, curated, normalise=same_gene_id):
    """Draft genes that are also in the curated model."""
    curated_ids = {normalise(g.id) for g in curated.genes}
    return {g.id for g in draft.genes if normalise(g.id) in curated_ids}


shared, rows = {}, []
for org in FILES:
    exact = shared_genes(ms[org], bigg[org], normalise=lambda gene_id: gene_id)
    shared[org] = shared_genes(ms[org], bigg[org])
    rows.append((org, len(exact), len(shared[org]),
                 100 * len(shared[org]) / len(ms[org].genes), 100 * len(shared[org]) / len(bigg[org].genes)))
pd.DataFrame(rows, columns=["organism", "shared (exact ids)", "shared (normalised ids)", "% of draft genes", "% of BiGG genes"]).set_index("organism").round(0)

In [ ]:
# ✅ Checkpoint
assert same_gene_id("BSU_00010") == same_gene_id("BSU00010") == "bsu00010", "same_gene_id() should remove '_' and lower-case"
assert len(shared["E. coli"]) > 1000, "most E. coli draft genes should be in iML1515"
print("✅ Gene overlap computed")

💬 Compare the two "shared" columns for *B. subtilis*: what would you have concluded with exact ids? And what does it mean that about a third of each draft's genes are **not** in the curated model?

## 7 · Your scorecard  ·  3 min

Everything together, in the layout of the scorecard slide. The cell also saves it as `my_scorecard.csv`: bring it to the 15:30 discussion.

In [ ]:
yes = lambda v: "yes" if v > 1e-6 else "no"
card = {}
for org in FILES:
    s_ms, s_bg = size(ms[org]), size(bigg[org])
    card[(org, "ModelSEED")] = [s_ms["all reactions"], s_ms["metabolites"], s_ms["genes"], len(gapfilled(ms[org])),
                                yes(growth_table.loc[org, "ModelSEED · Carbon-D-Glucose"]), len(shared[org])]
    card[(org, bigg[org].id)] = [s_bg["all reactions"], s_bg["metabolites"], s_bg["genes"], "–",
                                 yes(growth_table.loc[org, "BiGG · minimal glucose"]), "–"]
scorecard = pd.DataFrame(card, index=["All reactions", "Metabolites", "Genes", "Gap-filled reactions",
                                      "Grows on minimal glucose?", "Genes shared with BiGG"])
scorecard.to_csv("my_scorecard.csv")
scorecard

In [ ]:
# ✅ Checkpoint
assert Path("my_scorecard.csv").exists()
print("✅ Scorecard saved as my_scorecard.csv")

## Extension (if you finish early): does it make ATP from nothing?

This is the first check on the 15:30 checklist. Close every input, add a reaction that burns ATP, and maximise it. A model without energy-generating cycles must give exactly zero.

In [ ]:
def free_atp(model, atp_hydrolysis):
    with model:
        for r in model.boundary:
            r.bounds = (0, max(r.upper_bound, 0))      # nothing can enter the cell
        test = cobra.Reaction("ATP_TEST")
        model.add_reactions([test])
        test.build_reaction_from_string(atp_hydrolysis)
        model.objective = test
        return model.slim_optimize(error_value=0)


MS_ATP = "cpd00002_c0 + cpd00001_c0 --> cpd00008_c0 + cpd00009_c0 + cpd00067_c0"     # ATP + H2O -> ADP + Pi + H+
BIGG_ATP = "atp_c + h2o_c --> adp_c + pi_c + h_c"
pd.DataFrame({org: {"ModelSEED": free_atp(ms[org], MS_ATP), "BiGG": free_atp(bigg[org], BIGG_ATP)} for org in FILES}).T

All six models should give 0: no ATP from nothing. Keep one more number in mind for the discussion: how many reactions each model stores as running **only backwards** (lower bound below zero, upper bound zero).

In [ ]:
backwards = lambda m: sum(1 for r in m.reactions if not r.boundary and r.lower_bound < 0 and r.upper_bound <= 0)
pd.DataFrame({org: {"ModelSEED": backwards(ms[org]), "BiGG": backwards(bigg[org])} for org in FILES}).T

## Recap

| You compared | What it tells you |
|---|---|
| Identifiers (§2) | Two models can't be compared, or combined, until their names are translated |
| Size (§3) | Drafts can be bigger or smaller than curated models; size alone says little |
| Gap-filling (§4) | What the algorithm invented to make the model grow, and where to look first |
| Growth (§5) | Whether the model grows where it should, and not where it shouldn't (trpC2!) |
| Genes (§6) | How much of the curated knowledge the draft recovers, and how much it adds |

**At 15:30** we pool everyone's scorecards and ask how far an automated draft can be trusted, and what quality scores such as MEMOTE do and don't tell you.